# Ch. 7 · Audit de validation : fuites, groupes, incertitude, explicabilité

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch07-fiabilite-validation.ipynb)

**IA pour les sciences — M1.** Quatre expériences pour voir un modèle « tricher » et apprendre à le démasquer.

**Durée indicative :** 1 h 30 (atelier facultatif en autonomie)  
**Statut :** atelier facultatif hors volume horaire.

**Objectifs**

- Reproduire une fuite par sélection de variables hors validation croisée, puis la corriger.
- Montrer qu’une validation aléatoire surestime la performance quand les mesures sont groupées (réplicats).
- Choisir des métriques adaptées à des classes déséquilibrées.
- Quantifier l’incertitude d’une métrique de test par bootstrap.
- Utiliser l’importance par permutation et repérer une variable artefact.

**Prérequis :** chapitre 3 ; cours du chapitre 7.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## Expérience 1 — Une fuite qui fabrique un résultat à partir de rien

Données : **200 observations, 5 000 variables de bruit pur**, étiquettes aléatoires. Aucun modèle ne peut faire mieux que 50 %.
Erreur fréquente : sélectionner les 20 variables les plus corrélées à $y$ **sur tout le jeu**, puis faire une validation croisée.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=UserWarning)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
X_bruit = rng.normal(size=(200, 5000))
y_bruit = rng.integers(0, 2, 200)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# MAUVAIS : sélection sur tout le jeu, puis validation croisée
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_bruit, y_bruit)
fuite = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_bruit, cv=cv)

# BON : la sélection est réapprise dans chaque pli
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
correct = cross_val_score(pipe, X_bruit, y_bruit, cv=cv)
print(f'Avec fuite    : exactitude {fuite.mean():.2f}  <- illusion')
print(f'Sans fuite    : exactitude {correct.mean():.2f}  <- hasard, comme attendu')

**🔎 Pourquoi ?** Parmi 5 000 variables aléatoires, certaines sont corrélées à $y$ *par hasard*, y compris sur les futures observations de test.
Les choisir avec $y$ complet revient à regarder les réponses de l’examen. Règle : **toute étape apprise** (sélection, normalisation, imputation, ACP)
se place dans le `Pipeline`.

## Expérience 2 — Réplicats : validation aléatoire ou par groupe ?

40 échantillons biologiques, mesurés chacun **5 fois** (réplicats techniques). Chaque échantillon a une « signature » propre ;
la classe ne dépend que faiblement des mesures. Si des réplicats d’un même échantillon sont en entraînement et en test,
le modèle **reconnaît l’échantillon** au lieu d’apprendre la classe.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GroupKFold

n_ech, n_rep, p = 40, 5, 10
signature = rng.normal(size=(n_ech, p))
classe_ech = (signature[:, 0] + rng.normal(0, 2.0, n_ech) > 0).astype(int)
Xg = np.repeat(signature, n_rep, axis=0) + rng.normal(0, 0.2, size=(n_ech * n_rep, p))
yg = np.repeat(classe_ech, n_rep)
groupes = np.repeat(np.arange(n_ech), n_rep)

foret = RandomForestClassifier(n_estimators=300, random_state=0)
aleatoire = cross_val_score(foret, Xg, yg, cv=KFold(5, shuffle=True, random_state=0))
par_groupe = cross_val_score(foret, Xg, yg, cv=GroupKFold(5), groups=groupes)
print(f'Validation aléatoire (réplicats mélangés) : {aleatoire.mean():.2f}')
print(f'Validation par échantillon (GroupKFold)   : {par_groupe.mean():.2f}')

**🔎 À retenir.** L’**unité d’indépendance** est l’échantillon (ou le patient, le site, la campagne), pas la ligne du tableau.
La validation doit reproduire la situation d’usage : prédire pour un **nouvel** échantillon.

## Expérience 3 — Classes déséquilibrées : le piège de l’exactitude

In [ ]:
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier

X, y = make_classification(n_samples=600, n_features=12, n_informative=5,
                           weights=[0.9, 0.1], random_state=42)
modeles = {'Toujours majoritaire': DummyClassifier(strategy='most_frequent'),
           'Logistique': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))}
for nom, m in modeles.items():
    ligne = [nom]
    for metric in ['accuracy', 'balanced_accuracy', 'f1', 'recall']:
        s = cross_val_score(m, X, y, cv=cv, scoring=metric)
        ligne.append(f'{metric}={s.mean():.2f}')
    print(' | '.join(ligne))

Le classifieur « toujours majoritaire » obtient ~90 % d’exactitude… et un rappel nul : il ne détecte **aucun** cas positif.
Exactitude équilibrée, rappel et F1 révèlent ce qu’on cherche réellement.

## Expérience 4 — Combien vaut un score sur 50 observations ? Le bootstrap

On tire avec remise $n$ observations du test, on recalcule la métrique, on répète $B$ fois : la dispersion obtenue
estime l’incertitude due à la **taille finie** du test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

def ic_bootstrap(y_true, y_pred, metrique, B=2000, graine=0):
    r = np.random.default_rng(graine)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    vals = []
    for _ in range(B):
        idx = r.integers(0, len(y_true), len(y_true))
        vals.append(metrique(y_true[idx], y_pred[idx]))
    return np.percentile(vals, [2.5, 97.5])

modele = modeles['Logistique']
for taille_test in [50, 150, 300]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=taille_test, stratify=y, random_state=1)
    pred = modele.fit(X_tr, y_tr).predict(X_te)
    bas, haut = ic_bootstrap(y_te, pred, balanced_accuracy_score)
    print(f'test de {taille_test:3d} : exactitude équilibrée {balanced_accuracy_score(y_te, pred):.2f}  IC95 % [{bas:.2f}, {haut:.2f}]')

## Expérience 5 — Explicabilité ≠ causalité : la variable « plaque »

On ajoute une variable `plaque` : les positifs ont été, par organisation du laboratoire, mesurés surtout sur la plaque 2.
L’**importance par permutation** (baisse du score quand on mélange une variable) la classe en tête. Le modèle n’a pas découvert de biologie :
il a appris un **artefact de collecte**, qui disparaîtra dans un autre laboratoire.

In [ ]:
from sklearn.inspection import permutation_importance

plaque = np.where(y == 1, rng.choice([1, 2], len(y), p=[.1, .9]), rng.choice([1, 2], len(y), p=[.9, .1]))
X_art = np.column_stack([X, plaque])
noms = [f'mesure_{i}' for i in range(X.shape[1])] + ['plaque']
X_tr, X_te, y_tr, y_te = train_test_split(X_art, y, test_size=0.3, stratify=y, random_state=0)
foret = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_tr, y_tr)
imp = permutation_importance(foret, X_te, y_te, scoring='balanced_accuracy', n_repeats=20, random_state=0)
ordre = imp.importances_mean.argsort()[::-1][:8]
plt.barh([noms[i] for i in ordre][::-1], imp.importances_mean[ordre][::-1],
         xerr=imp.importances_std[ordre][::-1], color=['C3' if noms[i] == 'plaque' else 'C0' for i in ordre][::-1])
plt.xlabel('Baisse d’exactitude équilibrée après permutation'); plt.title('Importance par permutation'); plt.show()

### ✏️ Exercice 1 — Le modèle survit-il à un autre laboratoire ?

Simulez un « nouveau laboratoire » où la plaque est attribuée **au hasard** (indépendamment de y) sur les données de test. Comparez l’exactitude équilibrée de la forêt sur le test d’origine et sur ce nouveau test.

In [ ]:
X_te_nouveau = X_te.copy()
X_te_nouveau[:, -1] = ...      # plaques 1 ou 2 tirées au hasard
print('Test d’origine   :', round(balanced_accuracy_score(y_te, foret.predict(X_te)), 3))
print('Nouveau labo     :', round(balanced_accuracy_score(y_te, foret.predict(...)), 3))

<details><summary><b>Voir le corrigé</b></summary>

```python
X_te_nouveau = X_te.copy()
X_te_nouveau[:, -1] = rng.choice([1, 2], len(X_te))
print('Test d’origine   :', round(balanced_accuracy_score(y_te, foret.predict(X_te)), 3))
print('Nouveau labo     :', round(balanced_accuracy_score(y_te, foret.predict(X_te_nouveau)), 3))
```

La performance chute dès que l’artefact disparaît : un test réalisé dans les mêmes conditions de collecte que l’entraînement ne garantit pas la généralisation. Il faut un test **externe** (autre laboratoire, autre campagne).

</details>

### ✏️ Exercice 2 — Corriger la fuite de normalisation

Le code ci-dessous normalise tout le jeu `X_bruit` **avant** la validation croisée. Ici l’effet est faible, mais la pratique est fautive. Réécrivez-le avec un pipeline.

In [ ]:
# Version fautive
X_norm = StandardScaler().fit_transform(X_bruit[:, :50])
print(cross_val_score(LogisticRegression(max_iter=1000), X_norm, y_bruit, cv=cv).mean())
# Version correcte
pipe = ...
print(cross_val_score(pipe, X_bruit[:, :50], y_bruit, cv=cv).mean())

<details><summary><b>Voir le corrigé</b></summary>

```python
# Version fautive
X_norm = StandardScaler().fit_transform(X_bruit[:, :50])
print(cross_val_score(LogisticRegression(max_iter=1000), X_norm, y_bruit, cv=cv).mean())
# Version correcte
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print(cross_val_score(pipe, X_bruit[:, :50], y_bruit, cv=cv).mean())
```

La normalisation fuit peu (seulement une moyenne et un écart-type), contrairement à la sélection de variables qui utilise $y$. Le pipeline rend le protocole correct **par construction**, quelle que soit l’étape.

</details>

## Interprétation scientifique (à rédiger)

1. Classez les cinq expériences selon la gravité du biais qu’elles révèlent.
2. Quelle est l’unité d’indépendance dans vos propres données de recherche ?
3. Que signifie un intervalle de confiance large sur une métrique de test ?
4. Rédigez la section « limites » d’un article qui aurait utilisé la variable plaque sans la contrôler.